# ADAM


### 1️⃣ ¿Qué es Adam?

**Adam (Adaptive Moment Estimation)** es un algoritmo de optimización que extiende el descenso de gradiente combinando **dos ideas clave**:

1. **Momentum** → usa información histórica del gradiente
2. **Learning rate adaptativo** → ajusta automáticamente el tamaño del paso por parámetro

📌 En la práctica:

> **Adam es el optimizador por defecto en deep learning y ML moderno.**

---

### 2️⃣ Intuición (por qué funciona mejor)

Adam mantiene **dos promedios móviles** del gradiente:

* **Primer momento (media)** → dirección promedio
* **Segundo momento (varianza)** → magnitud / estabilidad

Esto permite:

* Avanzar rápido cuando el gradiente es consistente
* Reducir el paso cuando hay ruido u oscilaciones

![Image](https://machinelearningmastery.com/wp-content/uploads/2017/05/Comparison-of-Adam-to-Other-Optimization-Algorithms-Training-a-Multilayer-Perceptron.png?utm_source=chatgpt.com)

![Image](https://www.researchgate.net/publication/332715365/figure/fig2/AS%3A962461960241156%401606480221448/Adam-is-an-effective-gradient-descent-algorithm-for-ODEs-a-Using-a-constant-learning.png?utm_source=chatgpt.com)

![Image](https://i.sstatic.net/bWW3a.png?utm_source=chatgpt.com)

---

### 3️⃣ Formulación matemática (resumen operativo)

Para cada iteración ( t ):

1. Gradiente:
   $$
   g_t = \nabla J(\theta_t)
   $$

2. Momentos:
   $$
   m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t
   $$
   $$
   v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2
   $$

3. Corrección de sesgo:
   $$
   \hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad
   \hat{v}_t = \frac{v_t}{1 - \beta_2^t}
   $$

4. Actualización:
   $$
   \theta_{t+1} = \theta_t - \alpha \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}
   $$

**Valores estándar (industria):**

* ( $\alpha$ = 0.001 )
* ( $\beta_1$ = 0.9 )
* ( $\beta_2$ = 0.999 )
* ( $\epsilon$ = 10^{-8} )

> Estos valores rara vez se modifican.

---

### 4️⃣ Adam vs GD clásico

| Aspecto           | GD clásico | Adam         |
| ----------------- | ---------- | ------------ |
| Learning rate     | Fijo       | Adaptativo   |
| Sensible a escala | Sí         | No           |
| Oscilaciones      | Frecuentes | Amortiguadas |
| Convergencia      | Lenta      | Rápida       |
| Ajuste manual     | Alto       | Bajo         |

➡️ **Adam reduce drásticamente la necesidad de tuning**.

---

### 5️⃣ Implementación desde cero

´´´
def adam_optimizer(X, y, alpha=0.001, beta1=0.9, beta2=0.999, eps=1e-8, epochs=1000):
    m, n = X.shape
    theta = np.zeros((n, 1))
    m_t = np.zeros_like(theta)
    v_t = np.zeros_like(theta)
    cost_history = []

    for t in range(1, epochs + 1):
        grad = (1/m) * X.T @ (X @ theta - y)

        m_t = beta1 * m_t + (1 - beta1) * grad
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)

        m_hat = m_t / (1 - beta1 ** t)
        v_hat = v_t / (1 - beta2 ** t)

        theta -= alpha * m_hat / (np.sqrt(v_hat) + eps)

        cost = (1/(2*m)) * np.sum((X @ theta - y) ** 2)
        cost_history.append(cost)

    return theta, cost_history
´´´

### 6️⃣ Comparación empírica: GD vs Adam

```python
theta_gd, cost_gd = gradient_descent(X_b, y, alpha=0.05, epochs=500)
theta_adam, cost_adam = adam_optimizer(X_b, y, alpha=0.01, epochs=500)

plt.figure(figsize=(9,5))
plt.plot(cost_gd, label="Gradient Descent")
plt.plot(cost_adam, label="Adam")
plt.xlabel("Iteraciones")
plt.ylabel("Costo")
plt.title("GD vs Adam")
plt.legend()
plt.grid(True)
plt.show()
```

🔍 Observación típica:

* Adam **converge antes**
* Curva más suave
* Menor sensibilidad al learning rate

---

### 7️⃣ ¿Cuándo usar Adam en la práctica real?

✅ Usar Adam cuando:

* Dataset grande o ruidoso
* Features con distintas escalas
* Redes neuronales
* Entrenamiento continuo
* Prototipado rápido

⚠️ Considerar alternativas cuando:

* Dataset pequeño y bien condicionado → OLS
* Optimización convexa simple → GD clásico

---

### 8️⃣ Uso de Adam en librerías reales

```python
from sklearn.linear_model import SGDRegressor

model = SGDRegressor(
    loss="squared_error",
    learning_rate="adaptive",
    eta0=0.01
)
```

```python
# Keras / TensorFlow
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
```

```python
# PyTorch
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
```

📌 **Adam es estándar industrial**.

---

### 9️⃣ Resumen

> **Adam es descenso de gradiente inteligente**.

* Aprende rápido
* Se adapta automáticamente
* Reduce tuning manual
* Base del deep learning moderno

Si hoy entrenas un modelo **y no sabes qué optimizador usar** → **Adam**.





Si quieres, puedo:

* mostrar **casos donde Adam falla**
* comparar **Adam vs RMSProp vs SGD**
* explicar **por qué Adam generaliza distinto**
* integrar Adam en un **notebook docente evaluable**

¿Avanzamos con alguno?


## Casos donde ADAM falla

### 1️⃣ Adam converge… pero **generaliza peor** (overfitting implícito)

**Qué ocurre**

* Adam suele **minimizar la pérdida de entrenamiento muy rápido**.
* En ciertos problemas convexos o casi convexos, **SGD (con momentum)** obtiene **mejor desempeño en test**.

**Por qué**

* El learning rate adaptativo **reduce demasiado el paso** en algunos parámetros.
* Se llega a **mínimos “afilados”** (sharp minima) que generalizan peor.

**Síntomas**

* `loss_train ↓↓↓`
* `loss_val ↓` al inicio y luego **se estanca o empeora**
* Gap train–test mayor que con SGD

**Cuándo pasa**

* NLP y visión (clasificación)
* Datasets medianos/ruidosos
* Regularización débil

**Alternativa**

* **SGD + momentum** o **AdamW** (desacopla weight decay)

![Image](https://cdn.prod.website-files.com/62cd5ce03261cb3e98188470/62cd5ce03261cb541818860d_5f27fce80311fc570305e589_1%2Ag0ziB92FYavsqguqMqjS8w.png?utm_source=chatgpt.com)

![Image](https://moonlight-paper-snapshot.s3.ap-northeast-2.amazonaws.com/arxiv/a-method-for-enhancing-generalization-of-adam-by-multiple-integrations-0.png?utm_source=chatgpt.com)

![Image](https://machinelearningmastery.com/wp-content/uploads/2018/10/Line-Plots-of-Loss-on-Train-and-Test-Datasets-While-Training-Showing-an-Overfit-Model.png?utm_source=chatgpt.com)

---

### 2️⃣ Adam **no converge al óptimo** en problemas convexos simples

**Qué ocurre**

* En funciones convexas (p. ej., regresión lineal/logística), Adam puede **estancarse lejos del óptimo**.

**Por qué**

* La adaptación por parámetro **rompe propiedades de convergencia** teórica.
* El escalado por ( \sqrt{v_t} ) puede “congelar” parámetros.

**Síntomas**

* El costo baja rápido y luego **queda plano**
* SGD sigue mejorando; Adam no

**Cuándo pasa**

* Problemas convexos bien condicionados
* Features normalizados
* Objetivo simple (MSE, log-loss)

**Alternativa**

* **GD / SGD** con schedule de learning rate
* **LBFGS** (si el tamaño lo permite)

![Image](https://i.sstatic.net/pN62z.png?utm_source=chatgpt.com)

![Image](https://www.researchgate.net/publication/322383358/figure/fig2/AS%3A687015792226309%401540808737498/Train-error-and-loss-function-comparison-of-SGD-ADAM-and-LARS-for-ResNet-v2-model-on-the.ppm?utm_source=chatgpt.com)

![Image](https://miro.medium.com/v2/resize%3Afit%3A1400/1%2A6wJdRSAMmPyJAKI87bHKgQ.jpeg?utm_source=chatgpt.com)

---

### 3️⃣ Adam es **muy sensible a hiperparámetros** en ciertos contextos

**Qué ocurre**

* Aunque “parece” robusto, Adam **puede divergir** con:

  * learning rate apenas alto
  * gradientes ruidosos
  * mala normalización

**Síntomas**

* Oscilaciones
* Picos en la pérdida
* NaNs (especialmente en FP16)

**Cuándo pasa**

* Series temporales
* Datos con outliers
* Escalas muy dispares

**Alternativa**

* Bajar `lr`
* **Gradient clipping**
* **RMSProp** o **SGD + momentum**
* Normalización estricta

![Image](https://cdn-images-1.medium.com/max/800/1%2AEP8stDFdu_OxZFGimCZRtQ.jpeg?utm_source=chatgpt.com)

![Image](https://machinelearningmastery.com/wp-content/uploads/2017/05/Comparison-of-Adam-to-Other-Optimization-Algorithms-Training-a-Multilayer-Perceptron.png?utm_source=chatgpt.com)

![Image](https://i.sstatic.net/47T4x.jpg?utm_source=chatgpt.com)

---

### 4️⃣ Adam + weight decay “clásico” = **regularización incorrecta**

**Qué ocurre**

* En Adam, el weight decay tradicional **no es equivalente** a L2.
* Se regulariza “mal” y se pierde generalización.

**Síntomas**

* Peor test loss que SGD con L2
* Sensibilidad a `lambda`

**Cuándo pasa**

* Modelos grandes
* Regularización importante

**Alternativa**

* **AdamW** (weight decay desacoplado)
* Estándar actual en deep learning

![Image](https://www.fast.ai/images/adam_charts.png?utm_source=chatgpt.com)

![Image](https://miro.medium.com/1%2AMkKinCkJxO_8TMYVmK1E2A.png?utm_source=chatgpt.com)

---

### 5️⃣ Adam **no es ideal para fine-tuning final**

**Qué ocurre**

* Adam llega rápido “cerca” del mínimo, pero **no afina bien**.

**Síntomas**

* Últimos epochs sin mejora
* SGD sigue bajando la pérdida

**Práctica común**

1. Entrenar con **Adam**
2. Cambiar a **SGD + LR pequeño** para el ajuste final

**Alternativa**

* **Warmup + switch optimizer**
* Cosine annealing con SGD

![Image](https://miro.medium.com/v2/da%3Atrue/resize%3Afit%3A1200/0%2A_k3TqqWPBw8ccQlI?utm_source=chatgpt.com)

![Image](https://miro.medium.com/1%2A63HMdMyw_XDcNkRCQ1nrpw.png?utm_source=chatgpt.com)

![Image](https://miro.medium.com/v2/resize%3Afit%3A1400/0%2A_k3TqqWPBw8ccQlI?utm_source=chatgpt.com)

---

### 6️⃣ Resumen comparativo (decisión rápida)

| Escenario              | Adam | Alternativa    |
| ---------------------- | ---- | -------------- |
| Prototipado rápido     | ✅    | —              |
| Deep learning estándar | ✅    | AdamW          |
| Mejor generalización   | ⚠️   | SGD + momentum |
| Convexo simple         | ❌    | GD / LBFGS     |
| Fine-tuning final      | ⚠️   | SGD            |
| Regularización fuerte  | ❌    | AdamW          |

---

### 7️⃣ Regla práctica (docencia y producción)

> **Adam es excelente para llegar rápido, pero no siempre para llegar mejor.**

**Workflow recomendado**

1. Normalizar datos
2. Entrenar con **Adam/AdamW**
3. Monitorear validación
4. (Opcional) Cambiar a **SGD** para el ajuste final

---

